# 🐱🐶 Cat vs Dog — One Task Through the Whole Pipeline

**A deep-dive practical example, on real data.**

This notebook traces a *single* concrete task — deciding whether a photo is a **cat** or a **dog** — through the entire machine-learning pipeline, with real code and real numbers at every step.

### What kind of problem is this?
**Binary classification** (supervised learning). The output is a *category* — `cat` or `dog` — not a continuous number, so this is **not** a regression problem. (Linear regression predicts a quantity like a house price; here we predict a *label*. Note: *logistic regression* has "regression" in its name but is actually a classification method — it predicts the probability of a class.)

### The dataset is real and public
We use the **Kaggle "Dogs vs. Cats"** dataset — the images from the 2013 Kaggle competition *Dogs vs. Cats* (originally Microsoft's "Asirra" set), **25,000 labelled photos**. It loads in one line via TensorFlow Datasets (`cats_vs_dogs`) — no Kaggle account needed. A CIFAR-10 fallback is included.

> Competition page: https://www.kaggle.com/c/dogs-vs-cats  ·  Dataset card: https://www.tensorflow.org/datasets/catalog/cats_vs_dogs

### Each step maps to a topic

| Step | What we do | Topic |
|------|-----------|-------|
| 1 | Real image → feature vector | **Linear algebra** (vectors & matrices) |
| 2 | Reduce dimensions ("eigen-cats") | **Eigen analysis / PCA** |
| 3 | Inspect & scale the data | **EDA** (mean, std, balance, normalisation) |
| 4 | Pick a paradigm & fit a model | **Supervised learning + probability** |
| 5 | Classify a new image | **Bayes' rule + Gaussian density** |
| 6 | Hit overfitting, then upgrade | **SVM kernels → CNN → transfer learning** |
| 7 | Ship it | **Deployment / productionization** |

> **How to run:** `Runtime → Run all`. For Step 6 use a GPU runtime (`Runtime → Change runtime type → GPU`).

---

## 0. Setup & load the real dataset

`cats_vs_dogs` ships with TensorFlow Datasets (preinstalled in Colab). We load it, split off a test set, and keep a CPU-friendly subset for the classical-ML steps (1–5). The full pipeline is used for the CNN in Step 6.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

USE_CIFAR_FALLBACK = False   # set True to skip the ~800MB download and use CIFAR-10

print("TensorFlow:", tf.__version__)

In [ ]:
# --- Load real cat/dog photos ----------------------------------------------
# label convention everywhere in this notebook:  0 = cat, 1 = dog
if not USE_CIFAR_FALLBACK:
    import tensorflow_datasets as tfds
    # cats_vs_dogs has a single 'train' split (~23,262 valid images); we slice it.
    (raw_train, raw_test), info = tfds.load(
        "cats_vs_dogs",
        split=["train[:90%]", "train[90%:]"],
        as_supervised=True, with_info=True)
    print("Loaded Kaggle Dogs-vs-Cats:", info.splits["train"].num_examples, "images")
    SOURCE = "Kaggle Dogs vs. Cats"
else:
    from tensorflow.keras.datasets import cifar10
    (Xa, ya), (Xb, yb) = cifar10.load_data()
    CAT, DOG = 3, 5
    def to_ds(X, y):
        m = np.isin(y.ravel(), [CAT, DOG])
        imgs = X[m]; labs = (y.ravel()[m] == DOG).astype(np.int64)
        return tf.data.Dataset.from_tensor_slices((imgs, labs))
    raw_train, raw_test = to_ds(Xa, ya), to_ds(Xb, yb)
    SOURCE = "CIFAR-10 (cat & dog classes)"
    print("Using fallback:", SOURCE)

In [ ]:
# --- Materialise a small grayscale subset for the classical steps (1-5) -----
# We resize to a tiny grid and convert to grayscale so PCA / Bayes stay fast
# and interpretable.  These are REAL photos, just downscaled.
CLASSIC_PX = 32          # 32x32 grayscale -> 1024 features
N_CLASSIC  = 4000        # subset size for classical ML

def to_small_gray(ds, n):
    X, y = [], []
    for img, lab in ds.take(n):
        g = tf.image.rgb_to_grayscale(tf.image.resize(img, (CLASSIC_PX, CLASSIC_PX)))
        X.append(tf.squeeze(g).numpy())
        y.append(int(lab))
    return np.array(X, dtype="float32"), np.array(y, dtype=int)

imgs_tr, y_tr_full = to_small_gray(raw_train, N_CLASSIC)
imgs_te, y_te_full = to_small_gray(raw_test, N_CLASSIC // 4)
print("classical train:", imgs_tr.shape, " test:", imgs_te.shape, " source:", SOURCE)

---
## Step 1 — Raw data → feature vector (linear algebra)

A grayscale image is just a grid of brightness numbers (0 = black, 1 = white after scaling). To feed it to a model we **flatten** the 2-D grid into a single **feature vector** $\mathbf{x}$.

A $32\times32$ grayscale photo → a vector of $32\times32 = 1024$ numbers. Stack $N$ images and you get a **design matrix** $X$ of shape $N \times 1024$ — one row per image, one column per pixel. This matrix is what every later step operates on.

To make the idea vivid, we first shrink one real photo to a tiny **4×4** grid (the classic "16 pixels" illustration), then build the real $N\times1024$ matrix.

In [ ]:
# Tiny 4x4 illustration of "flatten an image into a vector", on a REAL photo
one = imgs_tr[0]
tiny = tf.image.resize(one[..., None], (4, 4)).numpy().squeeze()
print("A real photo shrunk to 4x4 brightness values (0-255):")
print(np.round(tiny).astype(int))
print("\nFlattened into a 16-number feature vector:")
print(np.round(tiny.reshape(-1)).astype(int))

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(one, cmap="gray"); ax[0].set_title(f"real photo {CLASSIC_PX}x{CLASSIC_PX} gray"); ax[0].axis("off")
ax[1].imshow(tiny, cmap="gray"); ax[1].set_title("same photo as 4x4"); ax[1].axis("off")
ax[2].bar(range(16), tiny.reshape(-1)); ax[2].set_title("its 16-number vector")
plt.tight_layout(); plt.show()

In [ ]:
# Build the real design matrix X (one row per image, one column per pixel)
D = CLASSIC_PX * CLASSIC_PX                  # 1024 features
X      = imgs_tr.reshape(len(imgs_tr), D) / 255.0     # scale 0-1
X_test = imgs_te.reshape(len(imgs_te), D) / 255.0
y, y_test = y_tr_full, y_te_full

print("Design matrix X:", X.shape, " (images x pixels)")
print("First image as a feature vector (first 12 of", D, "values):")
print(np.round(X[0][:12], 3))

---
## Step 2 — Reduce dimensions (eigen analysis / PCA)

1024 features is a lot, and neighbouring pixels are highly correlated. **Principal Component Analysis** finds the directions of greatest variance and lets us describe each image with far fewer numbers.

Pure linear algebra:

1. Center the data: $X_c = X - \bar{X}$.
2. Covariance matrix $C = \frac{1}{n-1} X_c^\top X_c$.
3. Eigen-decomposition $C\,\mathbf{v}_i = \lambda_i \mathbf{v}_i$ — eigenvectors are "directions", eigenvalues are how much variance lies along each.
4. Keep the top $K$ eigenvectors and project: each 1024-vector → a $K$-number vector.

On images, the top eigenvectors are literally ghostly cat/dog face templates — **"eigen-cats"**.

In [ ]:
from sklearn.decomposition import PCA

K = 50                                  # keep 50 principal components
pca = PCA(n_components=K, random_state=SEED).fit(X)
X_pca      = pca.transform(X)
X_test_pca = pca.transform(X_test)

ev = pca.explained_variance_ratio_
print("Top-5 components explain:", np.round(ev[:5], 3))
print(f"Top {K} components capture {ev.sum()*100:.1f}% of variance.")
print("Compression:", D, "features ->", K, "features")

# Visualise the leading eigenvectors ("eigen-cats")
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
axes[0].imshow(pca.mean_.reshape(CLASSIC_PX, CLASSIC_PX), cmap="gray")
axes[0].set_title("mean", fontsize=9); axes[0].axis("off")
for i in range(5):
    axes[i+1].imshow(pca.components_[i].reshape(CLASSIC_PX, CLASSIC_PX), cmap="gray")
    axes[i+1].set_title(f"PC {i+1}", fontsize=9); axes[i+1].axis("off")
plt.suptitle('"Eigen-cats" — the top variance directions of real photos')
plt.tight_layout(); plt.show()

In [ ]:
# Scree plot: how variance is distributed across components
plt.figure(figsize=(6, 3.5))
plt.plot(np.cumsum(ev), marker=".")
plt.xlabel("number of components"); plt.ylabel("cumulative variance explained")
plt.title("Scree / cumulative-variance plot"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

---
## Step 3 — Inspect first (EDA)

Before modelling, *look* at the data:

- **Class balance** — are cats and dogs roughly equal? (Imbalance biases models.)
- **Central tendency** — the **mean** of each class.
- **Dispersion** — the **standard deviation** (spread).
- **Scaling** — rescale features to a common range so no single feature dominates.

In [ ]:
# Class balance
n_cat = int((y == 0).sum()); n_dog = int((y == 1).sum())
print(f"Class balance:  cats = {n_cat}, dogs = {n_dog}  "
      f"({n_cat/len(y)*100:.0f}% / {n_dog/len(y)*100:.0f}%)")

# Central tendency & dispersion of overall brightness, per class
cat_b = X[y == 0].mean(axis=1); dog_b = X[y == 1].mean(axis=1)
print("\nMean brightness   cats: %.3f   dogs: %.3f" % (cat_b.mean(), dog_b.mean()))
print("Std  brightness   cats: %.3f   dogs: %.3f" % (cat_b.std(), dog_b.std()))

plt.figure(figsize=(6, 3.5))
plt.hist(cat_b, bins=30, alpha=.6, label="cat")
plt.hist(dog_b, bins=30, alpha=.6, label="dog")
plt.xlabel("per-image mean brightness"); plt.ylabel("count")
plt.title("EDA: brightness distribution by class (real photos)"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Scale the PCA features (standardise to mean 0, unit variance) so no
# single component dominates the distance / probability computations.
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_pca)
Xs      = scaler.transform(X_pca)
Xs_test = scaler.transform(X_test_pca)
print("After standardising the 50 PCA features:")
print("  feature means ~", np.round(Xs.mean(0)[:5], 2), "...")
print("  feature stds  ~", np.round(Xs.std(0)[:5], 2), "...")

---
## Step 4 — Pick a paradigm & fit a model (supervised learning)

We have **labels** (`cat`/`dog`), so this is **supervised classification**. We start with the simplest probabilistic classifier: **Gaussian (Naïve) Bayes**.

It models each class's features with a **Gaussian density**

$$p(\mathbf{x}\mid c) = \prod_j \frac{1}{\sqrt{2\pi\sigma_{cj}^2}}\exp\!\left(-\frac{(x_j-\mu_{cj})^2}{2\sigma_{cj}^2}\right)$$

and fits each $\mu_{cj}, \sigma_{cj}^2$ by **maximum likelihood** (= the sample mean and variance per class).

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

nb = GaussianNB().fit(Xs, y)            # MLE fit of per-class means & variances
print("Learned class priors  P(cat), P(dog):", np.round(nb.class_prior_, 3))
print("First 5 cat feature means:", np.round(nb.theta_[0][:5], 2))
print("First 5 dog feature means:", np.round(nb.theta_[1][:5], 2))

---
## Step 5 — Classify with Bayes' rule

For a new image $\mathbf{x}$, **Bayes' rule** combines the class likelihood with the prior:

$$P(\text{cat}\mid \mathbf{x}) = \frac{P(\mathbf{x}\mid \text{cat})\,P(\text{cat})}{P(\mathbf{x})}$$

The denominator is the same for both classes, so we compare numerators and pick the larger.

> *Lecture example:* if $P(\mathbf{x}\mid\text{cat})=0.04 > P(\mathbf{x}\mid\text{dog})=0.01$ with equal priors, predict **cat**.

In [ ]:
# Bayes' rule on one real test image, by hand
i = 0
log_joint = nb._joint_log_likelihood(Xs_test[i:i+1])[0]   # log[P(x|c) P(c)]
post = np.exp(log_joint - log_joint.max()); post /= post.sum()
print("Test image #%d  (true: %s)" % (i, "cat" if y_test[i] == 0 else "dog"))
print("  P(cat | x) = %.3f,  P(dog | x) = %.3f" % (post[0], post[1]))
print("  -> prediction:", "CAT" if post[0] > post[1] else "DOG")

In [ ]:
# Score Naive Bayes on the held-out test set
nb_tr = accuracy_score(y, nb.predict(Xs))
nb_te = accuracy_score(y_test, nb.predict(Xs_test))
print("Gaussian Naive Bayes  ->  train %.1f%%   test %.1f%%" % (nb_tr*100, nb_te*100))
print("Confusion matrix (rows=true, cols=pred) [cat, dog]:")
print(confusion_matrix(y_test, nb.predict(Xs_test)))

---
## Step 6 — Hit overfitting, then upgrade

Real photos are hard: a simple model that only sees compressed brightness features tops out low. The upgrade path:

1. **Kernel SVM** (RBF) on the PCA features — a curved decision boundary. Better, but still limited by hand-made features.
2. **CNN from scratch** — learns its own features from raw pixels via **backpropagation + Adam**, with **dropout** to curb overfitting.
3. **Transfer learning** (MobileNetV2 pretrained on ImageNet) — the *practical* approach professionals actually use, reaching ~**97–99%** on this dataset.

The lesson is the **ordering**: Naive Bayes < SVM < CNN < transfer learning.

In [ ]:
# 1) Kernel SVM on the SAME PCA features (apples-to-apples vs Naive Bayes)
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale"))
svm.fit(X_pca, y)
svm_te = svm.score(X_test_pca, y_test)
print("Kernel SVM (RBF) on PCA features -> test %.1f%%" % (svm_te*100))

In [ ]:
# 2) + 3) Build tf.data pipelines of full-resolution RGB photos for the CNNs
IMG_SIZE = 160
BATCH = 32

def prep(img, lab, training=False):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    if training:
        img = tf.image.random_flip_left_right(img)
    return tf.cast(img, tf.float32), lab

train_ds = (raw_train.map(lambda x, y: prep(x, y, True))
            .shuffle(1000).batch(BATCH).prefetch(tf.data.AUTOTUNE))
test_ds  = (raw_test.map(prep).batch(BATCH).prefetch(tf.data.AUTOTUNE))
print("Image pipelines ready at", IMG_SIZE, "x", IMG_SIZE)

In [ ]:
# 3) Transfer learning: MobileNetV2 backbone (frozen) + a small head
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                   include_top=False, weights="imagenet")
base.trainable = False                              # freeze pretrained features

inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = preprocess_input(inp)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)                          # dropout curbs overfitting
out = layers.Dense(1, activation="sigmoid")(x)      # P(dog)
cnn = Model(inp, out)
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = cnn.fit(train_ds, validation_data=test_ds, epochs=3)

In [ ]:
# Compare every model on the SAME real test set
cnn_te = cnn.evaluate(test_ds, verbose=0)[1]
print("MODEL COMPARISON  (real %s test set)" % SOURCE)
print("  Gaussian Naive Bayes      : %.1f%%   <- simple probabilistic baseline" % (nb_te*100))
print("  Kernel SVM (RBF, PCA)     : %.1f%%   <- classical ML, curved boundary" % (svm_te*100))
print("  Transfer-learning CNN     : %.1f%%   <- practical, production-grade" % (cnn_te*100))

plt.figure(figsize=(6, 3.5))
plt.plot(history.history["accuracy"], marker="o", label="train")
plt.plot(history.history["val_accuracy"], marker="o", label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title("Transfer-learning CNN")
plt.legend(); plt.tight_layout(); plt.show()

---
## Step 7 — Deploy (productionization)

A trained model is useless on your laptop. Deployment means: **save** the weights, wrap them in a **predict function**, and (in production) expose that behind an HTTP endpoint so any uploaded photo returns a label in milliseconds.

In [ ]:
import time

cnn.save("catdog_cnn.keras")                        # 1) persist the model
from tensorflow.keras.models import load_model
served = load_model("catdog_cnn.keras")             # 2) reload (fresh "server")

def predict(rgb_uint8):
    """rgb_uint8: HxWx3 array. Returns (label, probability)."""
    img = tf.image.resize(tf.cast(rgb_uint8, tf.float32), (IMG_SIZE, IMG_SIZE))
    p = float(served.predict(img[None, ...], verbose=0)[0, 0])   # P(dog)
    return ("dog" if p >= 0.5 else "cat"), (p if p >= 0.5 else 1 - p)

# 3) time one inference (the "milliseconds" claim) on a real test image
sample_img, sample_lab = next(iter(raw_test))
sample = sample_img.numpy()
t0 = time.time(); label, conf = predict(sample); ms = (time.time() - t0) * 1000

plt.imshow(sample.astype("uint8")); plt.axis("off")
plt.title(f"predicted {label} ({conf*100:.0f}%) in {ms:.0f} ms  |  true: "
          f"{'dog' if int(sample_lab) else 'cat'}")
plt.show()

In [ ]:
# What the production service looks like (FastAPI)
service_code = '''
# app.py  --  run with:  uvicorn app:app --reload
from fastapi import FastAPI, UploadFile
from tensorflow.keras.models import load_model
import tensorflow as tf
from PIL import Image
import numpy as np, io

app = FastAPI()
model = load_model("catdog_cnn.keras")          # load once at startup

@app.post("/predict")
async def predict(file: UploadFile):
    img = Image.open(io.BytesIO(await file.read())).convert("RGB").resize((160, 160))
    x = tf.cast(np.asarray(img), tf.float32)[None, ...]
    p = float(model.predict(x, verbose=0)[0, 0])
    return {"label": "dog" if p >= 0.5 else "cat",
            "confidence": p if p >= 0.5 else 1 - p}
'''
print(service_code)

---
## Recap — every step mapped to a topic

| Step | We did (on real Kaggle photos) | Topic |
|------|--------------------------------|-------|
| 1 | Flattened photos into vectors; stacked into an $N\times1024$ matrix | **Feature vectors & matrices (linear algebra)** |
| 2 | Eigen-decomposed the covariance, kept 50 components ("eigen-cats") | **PCA / eigen analysis** |
| 3 | Checked balance, mean & std, standardised features | **EDA + probability** |
| 4 | Labels ⇒ supervised; fit per-class Gaussians by MLE | **Supervised learning + Gaussian density** |
| 5 | Combined likelihood × prior, picked the larger | **Bayes' rule** |
| 6 | Naive Bayes → kernel SVM → transfer-learning CNN | **Classical ML + deep learning** |
| 7 | Saved the model, wrapped it in a `predict()` service | **Deployment / productionization** |

### Problem type, restated
This is **binary image classification** — *not* regression. We assign one of two discrete labels (`cat`/`dog`). Regression would instead predict a continuous number (e.g. an animal's weight from its photo).

### Why the accuracies climb
- **Naive Bayes / SVM** only see *compressed brightness* (PCA of grayscale) — they discard the spatial structure that distinguishes a cat's face from a dog's, so they stay modest.
- **The CNN** sees full-resolution colour and learns spatial features (edges → textures → ears/snouts).
- **Transfer learning** reuses features already learned from millions of ImageNet photos — which is exactly how practitioners hit ~97–99% on this dataset in practice.

### Try it yourself
- Set `USE_CIFAR_FALLBACK = True` to run instantly without the ~800MB download.
- In Step 2, lower `K` to 10 and watch the classical accuracies drop.
- In Step 6, unfreeze the top of `base` (`base.trainable = True`) and fine-tune for a few epochs — accuracy usually climbs another point or two.
- Remove the `Dropout` layer and watch train/validation curves diverge (overfitting).